<a href="https://colab.research.google.com/github/ChristinaCarvalho/Meter-Data-Prediction-AI-Project/blob/main/Meter_AI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collab resource monitoring

In [1]:
# resource monitoring
!free -h
!uptime

               total        used        free      shared  buff/cache   available
Mem:            12Gi       601Mi       8.9Gi       1.0Mi       3.2Gi        11Gi
Swap:             0B          0B          0B
 19:01:32 up  1:49,  0 users,  load average: 0.28, 0.27, 0.26


In [2]:
# Manually disables runtime
from google.colab import runtime
print('Do you want to unassign the runtime? (y/n)')
user_input = input()
if user_input == 'y':
    print(runtime.unassign())
else:
    print('Runtime still active.')

Do you want to unassign the runtime? (y/n)
None


# Extracting data and Forming Tensor

In [ ]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset, random_split
import ipywidgets as widgets
from IPython.display import display, clear_output
import calendar

filename = '/content/drive/MyDrive/AI_Meter_Project/Meter_data_test_for_Export.csv'

# Creates a pandas dataframe from CSV file
df = pd.read_csv(filename)
df = df[['CollectDts', 'MeterFlow']]

#Resample and round to 1-min intervals (Data is not perfectly alligned to 5 mins each time)
df['CollectDts'] = pd.to_datetime(df['CollectDts'], format='mixed').dt.round('1min')
df = df.set_index('CollectDts')
df_resampled = df[['MeterFlow']].resample('1min').mean()

#Print sample of data
print(df_resampled)

#Configuring data as a tensor
flow_data = torch.tensor(df_resampled['MeterFlow'].values, dtype=torch.float32)
print(flow_data)

# Plotting Initial Data

In [ ]:
available_months_ints = df_resampled.index.month.unique().tolist()
available_months_dict = {calendar.month_name[m]: m for m in available_months_ints}

# 3. Create Widgets
month_dd = widgets.Dropdown(options=list(available_months_dict.keys()), description='Month:', layout={'width': 'max-content'})
day_dd = widgets.Dropdown(options=['All'], description='Day:', layout={'width': 'max-content'})
hour_dd = widgets.Dropdown(options=['All'] + list(range(24)), description='Hour:', disabled=True, layout={'width': 'max-content'})
out_original = widgets.Output()

def update_dropdowns(*args):
    # Dynamically update the Day dropdown based on the Month selected
    month_int = available_months_dict[month_dd.value]
    max_days = calendar.monthrange(2026, month_int)[1] # Assuming year 2026 based on your data

    current_day = day_dd.value
    new_day_options = ['All'] + list(range(1, max_days + 1))
    day_dd.options = new_day_options

    # Retain selected day if it exists in the new month, otherwise reset to 'All'
    if current_day in new_day_options:
        day_dd.value = current_day
    else:
        day_dd.value = 'All'

    # Enable/Disable Hour dropdown
    if day_dd.value == 'All':
        hour_dd.value = 'All'
        hour_dd.disabled = True
    else:
        hour_dd.disabled = False

    plot_original_chart()

def plot_original_chart(*args):
    with out_original:
        clear_output(wait=True)
        month_int = available_months_dict[month_dd.value]
        day_val = day_dd.value
        hour_val = hour_dd.value

        # 1. Filter by Month
        filtered_data = df_resampled[df_resampled.index.month == month_int]
        title_suffix = f"{month_dd.value}"

        # 2. Filter by Day (if selected)
        if day_val != 'All':
            filtered_data = filtered_data[filtered_data.index.day == day_val]
            title_suffix += f" {day_val}"

            # 3. Filter by Hour (if selected)
            if hour_val != 'All':
                filtered_data = filtered_data[filtered_data.index.hour == hour_val]
                title_suffix += f" @ {hour_val}:00"

        if filtered_data.empty:
            print(f"No data available for {title_suffix}.")
            return

        missing_mask = filtered_data['MeterFlow'].isna()

        plt.figure(figsize=(15, 5))
        plt.plot(filtered_data.index, filtered_data['MeterFlow'], label='Original Meter Flow', color='black', linewidth=1.5)

        plt.fill_between(filtered_data.index, 0, 1, where=missing_mask,
                         color='red', alpha=0.4, transform=plt.gca().get_xaxis_transform(),
                         label='Missing Data (Nulls)')

        plt.title(f"Before Model: Original Data - {title_suffix}")
        plt.xlabel("Date & Time")
        plt.ylabel("Flow Rate")
        plt.legend(loc='upper right')
        plt.tight_layout()
        plt.show()

# Link observers to trigger updates when user clicks
month_dd.observe(update_dropdowns, names='value')
day_dd.observe(update_dropdowns, names='value')
hour_dd.observe(plot_original_chart, names='value')

# Setup initial UI
ui = widgets.HBox([month_dd, day_dd, hour_dd])
display(ui, out_original)
update_dropdowns() # Trigger the first draw

# Sliding Window

In [ ]:
# Extract the values as a 1D tensor
flow_data = torch.tensor(df['MeterFlow'].values, dtype=torch.float32)

# Sliding Window 
window_height = 35  # 15 mins past + 5 mins masked + 15 mins future
stride = 5

unfolded_data = flow_data.unfold(0, window_height, stride)

# CRITICAL STEP: When resampling, Pandas introduces NaNs for naturally missing data.
# Since we are intentionally masking data for training, we should only use
# windows that are fully populated (no NaNs) before we apply our artificial mask.
valid_mask = ~torch.isnan(unfolded_data).any(dim=1)
clean_windows = unfolded_data[valid_mask]

print(f"Total valid {window_height}-min windows extracted: {len(clean_windows)}")

# Masking the Data for Training

In [ ]:
class MaskedMeterDataset(Dataset):
    def __init__(self, windows, mask_size=5):
        self.windows = windows
        self.window_size = windows.shape[1]
        self.mask_size = mask_size

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        # Clone window to avoid modifying the original data in memory
        window = self.windows[idx].clone()

        # Calculate start and end indices for the 5-min mask in the exact middle
        # For a window of 35: Middle 5 are indices 15 to 19.
        mask_start = (self.window_size - self.mask_size) // 2
        mask_end = mask_start + self.mask_size

        # The target is the original unmasked 5-minute batch we are trying to predict
        target = window[mask_start:mask_end].clone()

        # Mask the input data (-1.0 so the model recognizes it as an anomaly/gap compared to normal, positive flow rates)
        window[mask_start:mask_end] = -1.0

        # Reshape window to add the feature dimension (seq_len) -> (35, 1)
        input_seq = window.unsqueeze(-1)

        return input_seq, target


# Create DataLoaders
dataset = MaskedMeterDataset(clean_windows, mask_size=5)

# 80/20 Train-Validation Split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Bidirectional LSTM Model

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout_prob=0.2):
        super(BiLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size * 2, output_size)
        self.attn = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attn(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        context = self.dropout(context)
        output = self.fc(context)
        return output

# Training the model

In [ ]:
# Initial Parameters
input_size = 1  # We only need 1 parameter since we are only predicting one value type
hidden_size = 64
output_size = 5  # Output is 5 since there are 5 data points in a batch

# Initialize the model, loss function, and optimizer
model = BiLSTM(input_size, hidden_size, output_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
num_epochs = 15
for epoch in range(num_epochs):
  model.train()
  total_loss = 0
  for batch_inputs, batch_targets in train_loader:
    optimizer.zero_grad()
    outputs = model(batch_inputs)
    loss = criterion(outputs, batch_targets)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  avg_train_loss = total_loss / len(train_loader)

  # Validation Loop
  model.eval()
  val_loss = 0
  with torch.no_grad():
      for val_inputs, val_targets in val_loader:
          val_outputs = model(val_inputs)
          v_loss = criterion(val_outputs, val_targets)
          val_loss += v_loss.item()

  avg_val_loss = val_loss / len(val_loader)

  print(f'Epoch [{epoch + 1}/{num_epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')

# Assigning Values after training

In [ ]:
df_filled = df_resampled.copy()

model.eval()

is_na = df_filled['MeterFlow'].isna()
na_groups = is_na.ne(is_na.shift()).cumsum()
missing_blocks = df_filled[is_na].groupby(na_groups)

count_filled = 0

with torch.no_grad(): 
    for _, block in missing_blocks:
        gap_timestamps = block.index

        # Process the block in chunks of 5 minutes max
        for i in range(0, len(gap_timestamps), 5):
            chunk_timestamps = gap_timestamps[i:i+5]

            # 15 mins past, 5 mins target (chunk), 14 mins future (total 35 mins)
            window_start = chunk_timestamps[0] - pd.Timedelta(minutes=15)
            window_end = chunk_timestamps[0] + pd.Timedelta(minutes=19)

            # Extract this full 35-minute window
            window_data = df_filled.loc[window_start:window_end, 'MeterFlow'].copy()

            # Ensure we actually successfully pulled exactly 35 minutes
            if len(window_data) == 35:
                window_data = window_data.interpolate(method='linear', limit_direction='both')
                # Fallback to 0 if the entire window was NaN (highly unlikely but safe)
                window_data = window_data.fillna(0)

                window_tensor = torch.tensor(window_data.values, dtype=torch.float32)

                # Mask the middle 5 minutes (indices 15 to 19)
                window_tensor[15:20] = -1.0

                # Reshape for the BiLSTM: [Batch Size=1, Sequence Length=35, Features=1]
                model_input = window_tensor.unsqueeze(0).unsqueeze(-1)

                prediction = model(model_input)
                predicted_values = prediction.squeeze().numpy()

                for j, ts in enumerate(chunk_timestamps):
                    df_filled.loc[ts, 'MeterFlow'] = predicted_values[j]
                    count_filled += 1

print(f"Filled {count_filled} missing values")

# Plotting The Data Afterwards

In [ ]:
month_dd_pred = widgets.Dropdown(options=list(available_months_dict.keys()), description='Month:', layout={'width': 'max-content'})
day_dd_pred = widgets.Dropdown(options=['All'], description='Day:', layout={'width': 'max-content'})
hour_dd_pred = widgets.Dropdown(options=['All'] + list(range(24)), description='Hour:', disabled=True, layout={'width': 'max-content'})
out_predicted = widgets.Output()

def update_pred_dropdowns(*args):
    month_int = available_months_dict[month_dd_pred.value]
    max_days = calendar.monthrange(2026, month_int)[1]

    current_day = day_dd_pred.value
    new_day_options = ['All'] + list(range(1, max_days + 1))
    day_dd_pred.options = new_day_options

    if current_day in new_day_options:
        day_dd_pred.value = current_day
    else:
        day_dd_pred.value = 'All'

    if day_dd_pred.value == 'All':
        hour_dd_pred.value = 'All'
        hour_dd_pred.disabled = True
    else:
        hour_dd_pred.disabled = False

    plot_predicted_chart()

def plot_predicted_chart(*args):
    with out_predicted:
        clear_output(wait=True)
        month_int = available_months_dict[month_dd_pred.value]
        day_val = day_dd_pred.value
        hour_val = hour_dd_pred.value

        orig_filtered = df_resampled[df_resampled.index.month == month_int]
        pred_filtered = df_filled[df_filled.index.month == month_int]

        title_suffix = f"{month_dd_pred.value}"

        if day_val != 'All':
            orig_filtered = orig_filtered[orig_filtered.index.day == day_val]
            pred_filtered = pred_filtered[pred_filtered.index.day == day_val]
            title_suffix += f" {day_val}"

            if hour_val != 'All':
                orig_filtered = orig_filtered[orig_filtered.index.hour == hour_val]
                pred_filtered = pred_filtered[pred_filtered.index.hour == hour_val]
                title_suffix += f" @ {hour_val}:00"

        if pred_filtered.empty:
            print(f"No data available for {title_suffix}.")
            return

        missing_mask = orig_filtered['MeterFlow'].isna()

        plt.figure(figsize=(15, 5))
        plt.plot(pred_filtered.index, pred_filtered['MeterFlow'], label='Predicted/Filled Flow', color='black', linewidth=1.5)

        plt.fill_between(pred_filtered.index, 0, 1, where=missing_mask,
                         color='green', alpha=0.4, transform=plt.gca().get_xaxis_transform(),
                         label='Predicted Data')

        plt.title(f"After Model: Predicted Data - {title_suffix}")
        plt.xlabel("Date & Time")
        plt.ylabel("Flow Rate")
        plt.legend(loc='upper right')
        plt.tight_layout()
        plt.show()

month_dd_pred.observe(update_pred_dropdowns, names='value')
day_dd_pred.observe(update_pred_dropdowns, names='value')
hour_dd_pred.observe(plot_predicted_chart, names='value')

ui_pred = widgets.HBox([month_dd_pred, day_dd_pred, hour_dd_pred])
display(ui_pred, out_predicted)
update_pred_dropdowns()